# Stage 3. Compute background functions

## 1. Setup

### 1.1 Imports

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from acid.config import load_config, prepare_output
from acid.image_processing.background.compute_background_function import (
    calculate_backgrounds,
    save_backgrounds,
    update_background_metadata,
)
from acid.utils.fov_axis_utils import get_fov_ch_shape
from acid.utils.metadata.filtering import filter_metadata_by_splits
from acid.utils.metadata.loading import load_metadata
from acid.utils.metadata.saving import save_metadata_dataframe


### 1.2 Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### 1.3 Load configuration

In [ ]:
CONFIG_FILE = None  # "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
prepare_output(config)
stage_cfg = config.background_function_calculation
paths_cfg = config.shared.paths

## 2. Input Data Preparation

In [ ]:
metadata_df, metadata_file_name = load_metadata(stage_cfg.metadata)
metadata_df = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=stage_cfg.dataset_split
).copy()
flag_column = stage_cfg.metadata.dataframe_columns.flag_column_name
metadata_df_ok = metadata_df.loc[
    metadata_df[flag_column] != config.quality_control.processing.flag_value
].copy()
if metadata_df_ok.empty:
    raise ValueError("No unflagged training images available for background estimation")
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(
    metadata_df_ok,
    stage_cfg.processing.fov_directory,
    fov_clm=stage_cfg.metadata.dataframe_columns.fov_column_name,
    channel_axis=stage_cfg.processing.channel_axis,
    null_value=stage_cfg.metadata.dataframe_columns.null_value,
)
Path(stage_cfg.image_saving.directory).mkdir(parents=True, exist_ok=True)

## 3. Background calculation
### 3.1 Helpers

### 3.1 Calculate and save the selected strategy

In [ ]:
backgrounds = calculate_backgrounds(metadata_df_ok, fov_shape, stage_cfg)
results = save_backgrounds(backgrounds, stage_cfg, config.project_identity.project_name)
metadata_df_updated = update_background_metadata(metadata_df, stage_cfg)
pd.DataFrame(results)

## 4. Save metadata

In [ ]:
metadata_path = save_metadata_dataframe(
    metadata_df_updated, stage_cfg.metadata, config.project_identity.project_name
)

## 5. Inspect results

In [ ]:
metadata_df_updated.head()

### 5.1 Display average backgrounds

In [ ]:
for group, background in backgrounds.items():
    fig, axes = plt.subplots(
        1, num_channels, figsize=(4 * num_channels, 4), squeeze=False
    )
    for channel, ax in enumerate(axes[0]):
        ax.imshow(
            np.take(background, channel, axis=stage_cfg.processing.channel_axis),
            cmap="gray",
        )
        ax.set_title(f"{group}: channel {channel}")
    plt.show()